# VeriqURL Full Pipeline Notebook

**Project:** VeriqURL — Lightweight URL-Based Phishing Detection and Awareness Tool  
**Purpose:** This notebook explains the full project pipeline from dataset preparation to model evaluation, threshold validation, explanation rules, and case study prediction.

This notebook is intended for project demonstration and supervisor review. It does not replace the production Streamlit app (`app.py`).

## 1. Project Setup

**Purpose:** Configure project paths and import the main Python libraries used throughout the project.

In [1]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd()
SRC_DIR = ROOT_DIR / "src"
RESULTS_DIR = ROOT_DIR / "results"
MODELS_DIR = ROOT_DIR / "models"
DATA_DIR = ROOT_DIR / "data"

sys.path.append(str(SRC_DIR))

print("Project root:", ROOT_DIR)
print("src exists:", SRC_DIR.exists())
print("results exists:", RESULTS_DIR.exists())
print("models exists:", MODELS_DIR.exists())

Project root: c:\Users\Wen Tao\Desktop\Studies\FYP\Phishing Detection\notebooks
src exists: False
results exists: False
models exists: False


## 2. Dataset Loading and Cleaning

**Purpose:** Show where the dataset is stored and how the cleaned dataset is loaded. The final system uses URL-only detection, so only URL text and labels are needed.

The cleaning script used in the project is `src/preprocess.py`.

In [ ]:
cleaned_path = DATA_DIR / "processed" / "kaggle_cleaned.csv"

if cleaned_path.exists():
    df = pd.read_csv(cleaned_path)
    print("Cleaned dataset shape:", df.shape)
    display(df.head())
else:
    print("Cleaned dataset not found. Run: python src/preprocess.py")

## 3. Train / Validation / Test Split

**Purpose:** Load the generated train, validation, and test splits. These splits are used consistently for Random Forest and TCN evaluation.

The split script used in the project is `src/split_data.py`.

In [ ]:
split_dir = DATA_DIR / "processed" / "splits"
for name in ["train", "val", "test"]:
    path = split_dir / f"{name}.csv"
    if path.exists():
        split_df = pd.read_csv(path)
        print(f"{name}: {split_df.shape}")
        if "label" in split_df.columns:
            print(split_df["label"].value_counts().to_dict())
    else:
        print(f"Missing: {path}")

## 4. Random Forest Feature Extraction

**Purpose:** Demonstrate the handcrafted lexical and structural URL features used by the Random Forest model.

The feature extraction script used in the project is `src/feature_extraction.py`. The deployed app uses RF-compatible features through `extract_model_features()` in `src/explanation_rules.py` to ensure the prediction input matches the trained model.

In [ ]:
from explanation_rules import extract_model_features, extract_explanation_indicators

sample_url = "http://secure-paypal-login-verification.com/account/verify"
features = extract_model_features(sample_url)
print("RF-compatible model features:")
display(pd.DataFrame([features]).T.rename(columns={0: "value"}))

## 5. Random Forest Training

**Purpose:** Explain how the Random Forest baseline was trained. Random Forest is the lightweight feature-based baseline model.

The full training script is `src/train_rf.py`. If the model already exists, this notebook does not need to retrain it.

In [ ]:
rf_model_path = MODELS_DIR / "random_forest_model.pkl"
print("RF model exists:", rf_model_path.exists())
print("RF model path:", rf_model_path)

# To retrain manually, run this in terminal:
# python src/train_rf.py

## 6. TCN Character Vocabulary and URL Encoding

**Purpose:** Show how URLs are transformed into fixed-length character-level sequences for the TCN model.

The TCN data preparation script is `src/prepare_tcn_data.py`.

In [ ]:
vocab_path = DATA_DIR / "processed" / "tcn" / "char_vocab.json"

if vocab_path.exists():
    with open(vocab_path, "r", encoding="utf-8") as f:
        char_to_idx = json.load(f)
    print("Vocabulary size:", len(char_to_idx))
    print("First 20 vocabulary items:", list(char_to_idx.items())[:20])
else:
    print("char_vocab.json not found. Run: python src/prepare_tcn_data.py")

In [ ]:
from predict_compare import encode_url_for_tcn

if vocab_path.exists():
    encoded = encode_url_for_tcn(sample_url, char_to_idx, max_len=200)
    print("Encoded sequence length:", len(encoded))
    print("Non-padding character count:", int(np.count_nonzero(encoded)))
    print("First 60 encoded values:", encoded[:60])

## 7. TCN Model Architecture

**Purpose:** Show the TCN architecture used in the final implementation. The model uses character embedding, temporal convolutional blocks, global max pooling, and a fully connected output layer.

The training script is `src/train_tcn.py`. Full TCN retraining is not run inside this notebook because it can take a long time.

In [ ]:
from predict_compare import TCNClassifier

if vocab_path.exists():
    model = TCNClassifier(
        vocab_size=len(char_to_idx),
        embed_dim=32,
        num_channels=64,
        kernel_size=3,
        dropout=0.2,
    )
    print(model)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Trainable parameters:", total_params)

## 8. TCN Training Summary

**Purpose:** Display saved TCN training history and summary. This provides evidence of the training process without retraining the model in the notebook.

In [ ]:
history_path = RESULTS_DIR / "tcn_training_history.csv"
summary_path = RESULTS_DIR / "tcn_training_summary.csv"

if history_path.exists():
    history_df = pd.read_csv(history_path)
    print("TCN training history:")
    display(history_df.tail())
else:
    print("tcn_training_history.csv not found.")

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
    print("TCN training summary:")
    display(summary_df)
else:
    print("tcn_training_summary.csv not found.")

## 9. Model Evaluation

**Purpose:** Load the saved model evaluation results for Random Forest and TCN. These metrics are reported in the final report.

In [ ]:
for filename in ["rf_metrics.csv", "tcn_metrics.csv", "tcn_confusion_matrix.csv"]:
    path = RESULTS_DIR / filename
    print("Displaying:", filename)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Missing:", path)

## 10. Model Comparison and Lightweight Metrics

**Purpose:** Compare RF and TCN using both classification metrics and lightweight efficiency metrics such as TES, IES, and RTDE.

In [ ]:
comparison_path = RESULTS_DIR / "model_comparison.csv"
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    display(comparison_df)
else:
    print("model_comparison.csv not found. Run: python src/compare_models.py")

## 11. Risk Threshold Validation

**Purpose:** Explain why the High Risk threshold is set to 0.80. Candidate thresholds were tested on the validation set to compare High Risk precision, recall, false positives, and Suspicious category size.

The validation script is `src/validate_risk_thresholds.py`.

In [ ]:
threshold_path = RESULTS_DIR / "risk_threshold_validation.csv"
distribution_path = RESULTS_DIR / "risk_threshold_distribution.csv"

if threshold_path.exists():
    threshold_df = pd.read_csv(threshold_path)
    display(threshold_df)
else:
    print("risk_threshold_validation.csv not found. Run: python src/validate_risk_thresholds.py")

if distribution_path.exists():
    print("Risk distribution by threshold:")
    display(pd.read_csv(distribution_path))

## 12. Evidence-based Explanation Layer

**Purpose:** Demonstrate how VeriqURL extracts explanation indicators from a URL. These indicators are used to generate human-readable reasons and evidence-based awareness guidance.

In [ ]:
indicators = extract_explanation_indicators(sample_url)
print("Explanation indicators for:", sample_url)
display(pd.DataFrame([indicators]).T.rename(columns={0: "value"}))

## 13. Case Study Prediction

**Purpose:** Run the final prediction pipeline on representative benign, phishing, and suspicious URLs. This demonstrates the same logic used by the Streamlit app.

In [ ]:
import torch
from predict_compare import (
    load_rf_model,
    load_char_vocab,
    load_tcn_model,
    predict_rf,
    predict_tcn,
    get_final_decision,
)
from explanation_rules import build_explanation_result

case_urls = [
    "https://www.google.com",
    "http://secure-paypal-login-verification.com/account/verify",
    "https://update-service-login.example.org",
]

if rf_model_path.exists() and (MODELS_DIR / "tcn_model.pt").exists() and vocab_path.exists():
    rf_model = load_rf_model()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    char_to_idx = load_char_vocab()
    tcn_model = load_tcn_model(len(char_to_idx), device)

    rows = []
    for url in case_urls:
        rf_result = predict_rf(url, rf_model)
        tcn_result = predict_tcn(url, tcn_model, char_to_idx, device)
        final_decision = get_final_decision(rf_result, tcn_result)
        explanation = build_explanation_result(
            url,
            max(rf_result["phishing_probability"], tcn_result["phishing_probability"])
        )
        rows.append({
            "url": url,
            "rf_label": rf_result["predicted_label"],
            "rf_phishing_probability": round(rf_result["phishing_probability"], 4),
            "rf_risk": rf_result["risk_level"],
            "tcn_label": tcn_result["predicted_label"],
            "tcn_phishing_probability": round(tcn_result["phishing_probability"], 4),
            "tcn_risk": tcn_result["risk_level"],
            "average_phishing_probability": round(final_decision["average_phishing_probability"], 4),
            "final_label": final_decision["final_label"],
            "final_risk": final_decision["final_risk"],
            "top_reason": explanation["reasons"][0],
            "top_awareness_guidance": explanation["awareness_guidance"][0],
        })
    display(pd.DataFrame(rows))
else:
    print("Model files are missing. Case study prediction requires saved model artifacts.")

## 14. Summary

**Purpose:** Summarise what the pipeline demonstrates.

This notebook shows the complete VeriqURL workflow: dataset preparation, RF feature extraction, TCN sequence encoding, model training artefacts, evaluation metrics, lightweight comparison, threshold validation, explanation indicators, and case study prediction. The deployed Streamlit app uses the saved model artefacts and explanation rules to provide URL analysis and awareness guidance to users.